This code, trained on Kaggle's Titanic dataset, returns the survival probability for each of the following factors: 
1. Age, 2. Gender, and 3. Cabin Class. 

If a negative number is entered for age, the program will terminate.

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import numpy as np
import os
import zipfile

# -----------------------------------------------------------------------------
# 1. Google Colab Environment Setup and Data Preparation
# -----------------------------------------------------------------------------

# Settings to use Kaggle API in Google Colab environment
# Running this cell will show a file upload dialog.
# Please upload the kaggle.json file saved on your PC.
try:
    from google.colab import files
    print("Running in Google Colab environment. Please upload the kaggle.json file.")

    # Delete kaggle.json file if it already exists before uploading again
    if os.path.exists('kaggle.json'):
        os.remove('kaggle.json')

    uploaded = files.upload()

    # Check if file was uploaded
    if 'kaggle.json' not in uploaded:
        raise FileNotFoundError("kaggle.json file was not uploaded.")

    # Configure Kaggle API
    if not os.path.exists('/root/.kaggle'):
        os.makedirs('/root/.kaggle')

    os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 600)
    print("Kaggle API setup complete.")

    # Install kaggle library
    !pip install -q kaggle

except ImportError:
    # For non-Colab environments
    print("Not in Google Colab environment. Please check your local setup.")
    pass

import kaggle

# Download dataset using Kaggle API
competition_name = 'titanic'
data_file = 'train.csv'
zip_file_path = f'{competition_name}.zip'

# Download from Kaggle if train.csv does not exist
if not os.path.exists(data_file):
    print(f"'{data_file}' not found. Downloading data from Kaggle...")
    try:
        # Authenticate Kaggle API
        kaggle.api.authenticate()
        # Download data
        kaggle.api.competition_download_files(competition_name, path='.')

        # Extract zip file
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            zip_ref.extractall('.')

        # Delete downloaded zip file
        os.remove(zip_file_path)
        print("Data download and extraction complete.")
    except Exception as e:
        print(f"An error occurred while using the Kaggle API: {e}")
        print("Please check if your API token is set correctly and if you have joined the competition.")
        # Create temporary DataFrame for example execution in case of error
        data = pd.DataFrame({
            'PassengerId': range(1, 892), 'Survived': np.random.randint(0, 2, 891),
            'Pclass': np.random.randint(1, 4, 891), 'Name': ['' for _ in range(891)],
            'Sex': np.random.choice(['male', 'female'], 891), 'Age': np.random.uniform(1, 80, 891),
            'SibSp': np.random.randint(0, 5, 891), 'Parch': np.random.randint(0, 5, 891),
            'Ticket': ['' for _ in range(891)], 'Fare': np.random.uniform(0, 500, 891),
            'Cabin': ['' for _ in range(891)], 'Embarked': np.random.choice(['S', 'C', 'Q'], 891)
        })
        data['Age'].iloc[np.random.choice(data.index, 100)] = np.nan
        print("\nRunning example with temporary data.")
        data.to_csv(data_file, index=False) # Create temporary file for subsequent logic to work

# Load data
data = pd.read_csv(data_file)


# -----------------------------------------------------------------------------
# 2. Data Preprocessing
# -----------------------------------------------------------------------------

# Separate features (X) and target (y)
X = data.drop('Survived', axis=1)
y = data['Survived']

# Define numerical and categorical features
numeric_features = ['Age', 'Fare', 'SibSp', 'Parch']
categorical_features = ['Pclass', 'Sex', 'Embarked']

# Create preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)])

# Execute data preprocessing
X_processed = preprocessor.fit_transform(X)
y_processed = y.values

# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_processed, y_processed, test_size=0.2, random_state=42, stratify=y_processed)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train.toarray() if hasattr(X_train, "toarray") else X_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val.toarray() if hasattr(X_val, "toarray") else X_val, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

# Create DataLoaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=32, shuffle=False)

# Check the number of input features
input_features = X_train_tensor.shape[1]
print(f"\nNumber of input features after preprocessing: {input_features}")


# -----------------------------------------------------------------------------
# 3. Define Model (inheriting nn.Module)
# -----------------------------------------------------------------------------
class SimpleClassifier(nn.Module):
    def __init__(self, num_features):
        super(SimpleClassifier, self).__init__()
        self.layer_1 = nn.Linear(num_features, 64)
        self.layer_2 = nn.Linear(64, 32)
        self.layer_out = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.2)
        self.batchnorm1 = nn.BatchNorm1d(64)
        self.batchnorm2 = nn.BatchNorm1d(32)

    def forward(self, inputs):
        x = self.relu(self.layer_1(inputs))
        x = self.batchnorm1(x)
        x = self.relu(self.layer_2(x))
        x = self.batchnorm2(x)
        x = self.dropout(x)
        x = torch.sigmoid(self.layer_out(x))
        return x

# -----------------------------------------------------------------------------
# 4. Model Training
# -----------------------------------------------------------------------------
epochs = 100
learning_rate = 0.001
model = SimpleClassifier(num_features=input_features)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print("\nStarting model training...")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_idx, (data_batch, targets_batch) in enumerate(train_loader):
        outputs = model(data_batch)
        loss = criterion(outputs, targets_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(train_loader):.4f}')

# -----------------------------------------------------------------------------
# 5. Model Evaluation
# -----------------------------------------------------------------------------
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for data_batch, targets_batch in val_loader:
        outputs = model(data_batch)
        predicted = (outputs.data > 0.5).float()
        total += targets_batch.size(0)
        correct += (predicted == targets_batch).sum().item()

print(f'\nValidation Accuracy: {100 * correct / total:.2f} %')

# -----------------------------------------------------------------------------
# 6. Real-time Prediction based on User Input
# -----------------------------------------------------------------------------
print("\n--- Survival Probability Prediction ---")
print("Enter a negative number (-1) for Age to exit.")

# Fill other numerical features needed for prediction with median values from the training data.
fare_median = data['Fare'].median()
sibsp_median = data['SibSp'].median()
parch_median = data['Parch'].median()
embarked_most_frequent = data['Embarked'].mode()[0]


while True:
    try:
        # Get user input
        input_age = int(input("\nEnter Age: "))
        if input_age < 0:
            break

        input_sex = input("Enter Sex (male/female): ").lower()
        if input_sex not in ['male', 'female']:
            print("Sex can only be 'male' or 'female'.")
            continue

        input_pclass = int(input("Enter Passenger Class (1/2/3): "))
        if input_pclass not in [1, 2, 3]:
            print("Passenger Class can only be 1, 2, or 3.")
            continue

        # Convert input to DataFrame
        # Ensure the column order is the same as the preprocessor was trained on.
        user_input = pd.DataFrame({
            'Age': [input_age],
            'Fare': [fare_median],
            'SibSp': [sibsp_median],
            'Parch': [parch_median],
            'Pclass': [input_pclass],
            'Sex': [input_sex],
            'Embarked': [embarked_most_frequent]
        })

        # Preprocess data
        processed_input = preprocessor.transform(user_input)

        # Convert to tensor (corrected error)
        # Handle flexibility based on preprocessor output type (sparse matrix or numpy array)
        input_tensor = torch.tensor(processed_input.toarray() if hasattr(processed_input, "toarray") else processed_input, dtype=torch.float32)

        # Model prediction
        model.eval()
        with torch.no_grad():
            prediction_prob = model(input_tensor)

        print(f"-> Survival probability for the entered information is {prediction_prob.item() * 100:.2f}%.")

    except ValueError:
        print("Invalid input. Please ensure you enter numbers where required.")
    except Exception as e:
        print(f"An error occurred: {e}")

print("\nExiting program.")